In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import time
from multiprocessing import Pool
from tqdm.auto import tqdm
import re
from copy import deepcopy

import numpy as np
from scipy import integrate
from matplotlib import pyplot as plt

import noctiluca as nl
import bayesmsd

/home/sgh/gitlibs/chromatin_dynamics/.venv_SD_py39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
filename = '/data/sgh/science/2024_minflux/20260106_chromatin_dynamics_all_data.h5'
data       = nl.io.load.hdf5(filename)['data']

In [3]:
n_subsample = 4 # cut off the "kink" at the beginning of MINFLUX data
def subsample(traj):
    out = nl.Trajectory(traj[::n_subsample])
    out.meta['Δt'] = n_subsample*traj.meta['Δt']
    return out

In [4]:
data.makeSelection('minflux')
data.apply(subsample, inplace=True)

# Fits

In [5]:
def chop(traj, dt=None, L=200, Fmin=2):
    if dt is None:
        dt = traj.meta['Δt']
    
    def chop_traj(traj, dt=dt):
        if 'Δt' in traj.meta:
            dt = traj.meta['Δt']
            
        chops = []
        i0 = 0
        while i0 < len(traj):
            i1 = i0+L
            chop = traj.data[:, i0:min(i1, len(traj)), :]
            try:
                t_start = np.nonzero(~np.any(np.isnan(chop), axis=(0, 2)))[0][0]
            except IndexError: # no valid entries in this chop
                new_traj = nl.Trajectory(chop[:, [0]])
            else:
                new_traj = nl.Trajectory(chop[:, t_start:])
                
            new_traj.meta['Δt'] = dt
            chops.append(new_traj)

            i0 = i1
            
        return chops
    
    chops = chop_traj(traj)
    out = nl.TaggedSet(chops, hasTags=False)
    while len(chops) > 1:
        cg_traj = nl.Trajectory(np.stack([traj.data[:, 0] for traj in chops], axis=1))
        cg_traj.meta['Δt'] = L*chops[0].meta['Δt']
        chops = chop_traj(cg_traj)
        for traj in chops:
            out.add(traj)
    
    # Clean out useless trajectories
    out.makeSelection(lambda traj, _: traj.F < Fmin)
    out.deleteSelection()
    return out

In [6]:
ct = 'RH30'
bar = tqdm()

fits = {}
for treatment in ['ctrl']:

    cond = ['H2B', ct, treatment]
    fits[treatment] = {
        'single' : {},
        'joints' : {},
    }

    # Minflux
    data.makeSelection(['minflux', *cond], logic=all)
    dt = data[0].meta['Δt']

    fitdata = nl.TaggedSet()
    for traj in data:
        fitdata |= chop(traj.rescale(1e6, keepmeta=['Δt']))

    with nl.Parallelize():
        _ = nl.analysis.MSD(fitdata, chunksize=10, show_progress=True)

    fit = bayesmsd.lib.NPFit(fitdata, motion_blur_f=dt/n_subsample, parametrization='(log(αΓ), α)')
    fit.parameters['log(σ²) (dim 1)'].fix_to = 'log(σ²) (dim 0)'
    fit.likelihood_chunksize = 200

    fits[treatment]['single'][f'minflux'] = fit

    bar.update()

    # Conventional
    for dt_tag in ['100ms', '2s']:
        data.makeSelection(['SPT', dt_tag, *cond], logic=all)
        dt = data[0].meta['Δt']
        tau_e = 0.08671 # same exposure for both conditions

        fitdata = data.apply(lambda traj : traj.relative(keepmeta=['MSD', 'Δt']), inplace=False)

        fit = bayesmsd.lib.NPFit(fitdata, motion_blur_f=tau_e, parametrization='(log(αΓ), α)')
        fit.parameters['log(σ²) (dim 1)'].fix_to = 'log(σ²) (dim 0)'
        fit.likelihood_chunksize = 100

        fits[treatment]['single'][f'SPT-{dt_tag}'] = fit

        bar.update()

    # Assemble list of fit(group)s to run
    groups = {
        'minflux'       : ['minflux'],
        'SPT 100ms'     : ['SPT-100ms'],
        'SPT 2s'        : ['SPT-2s'],
        'SPT'           : ['SPT-100ms', 'SPT-2s'],
        'minflux + SPT' : ['minflux', 'SPT-100ms', 'SPT-2s'],
    }

    for groupname in groups:
        fits_dict = fits[treatment]['single']

        fit = bayesmsd.FitGroup({name : fits_dict[name] for name in groups[groupname]})
        fit.parameters['α']       = deepcopy(fits_dict['minflux'].parameters[      'α (dim 0)'])
        fit.parameters['log(αΓ)'] = deepcopy(fits_dict['minflux'].parameters['log(αΓ) (dim 0)'])

        # hacky...
        def patch_initial_params(self=fit):
            params = type(self).initial_params(self)
            a    = [val for key, val in params.items() if      'α' in key][0]
            logG = [val for key, val in params.items() if 'log(αΓ)' in key][0]
            params['α'] = a
            params['log(αΓ)'] = logG
            return params
        fit.initial_params = patch_initial_params

        for fitname in fit.fits_dict:
            fit.parameters[fitname+f' α (dim 0)'].fix_to = 'α'
            if fitname == 'minflux':
                fit.parameters[fitname+f' log(αΓ) (dim 0)'].fix_to = 'log(αΓ)'
            else: # not minflux, so correct for 2-loc
                def twoGref(params): return params['log(αΓ)']+np.log(2)
                fit.parameters[fitname+f' log(αΓ) (dim 0)'].fix_to = twoGref

        fits[treatment]['joints'][groupname] = fit

        bar.update()

bar.close()

0it [00:00, ?it/s]
100%|█████████████████████████████████████████████████████████████████████████████████████████| 2848/2848 [00:01<00:00, 2405.70it/s]
8it [00:05,  1.46it/s]


In [7]:
fitres = {}
for treatment in ['ctrl']:
    print()
    print(17*'=')
    print(f'|| {ct:>5s} {treatment:<5s} ||')
    print(17*'=')
    print()
    
    fitres[treatment] = {}
    for name in fits[treatment]['joints']:
        print(name)
        print('='*20)

        with nl.Parallelize():
            fitres[treatment][name] = fits[treatment]['joints'][name].run(show_progress=True)

        for key in fitres[treatment][name]['params']:
            print(key, fitres[treatment][name]['params'][key])
        print()


||  RH30 ctrl  ||

minflux


fit iterations: 75it [00:33,  2.23it/s]


minflux log(σ²) (dim 0) -8.339842126695693
α 0.26568573669531764
log(αΓ) -7.055289738446303
minflux α (dim 0) 0.26568573669531764
minflux log(αΓ) (dim 0) -7.055289738446303

SPT 100ms


fit iterations: 68it [00:27,  2.48it/s]


SPT-100ms log(σ²) (dim 0) -7.033681832683554
α 0.3558337417517671
log(αΓ) -6.568353989554904
SPT-100ms α (dim 0) 0.3558337417517671
SPT-100ms log(αΓ) (dim 0) -5.8752068089949585

SPT 2s


...

SPT-2s log(σ²) (dim 0) -6.142154730176287
α 0.5406073136793024
log(αΓ) -7.0271887858811874
SPT-2s α (dim 0) 0.5406073136793024
SPT-2s log(αΓ) (dim 0) -6.334041605321242

SPT


fit iterations: 73it [00:45,  1.62it/s]


SPT-100ms log(σ²) (dim 0) -7.035206275928942
SPT-2s log(σ²) (dim 0) -7.8198555931758715
α 0.34060833376987826
log(αΓ) -6.614295315665519
SPT-100ms α (dim 0) 0.34060833376987826
SPT-2s α (dim 0) 0.34060833376987826
SPT-100ms log(αΓ) (dim 0) -5.9211481351055735
SPT-2s log(αΓ) (dim 0) -5.9211481351055735

minflux + SPT


fit iterations: 172it [02:30,  1.14it/s]

minflux log(σ²) (dim 0) -8.218879431233601
SPT-100ms log(σ²) (dim 0) -7.015513545543207
SPT-2s log(σ²) (dim 0) -7.687593873361798
α 0.3456841861947864
log(αΓ) -6.623713583363033
minflux α (dim 0) 0.3456841861947864
minflux log(αΓ) (dim 0) -6.623713583363033
SPT-100ms α (dim 0) 0.3456841861947864
SPT-2s α (dim 0) 0.3456841861947864
SPT-100ms log(αΓ) (dim 0) -5.9305664028030876
SPT-2s log(αΓ) (dim 0) -5.9305664028030876



In [8]:
nl.io.write.hdf5(fitres, f'/data/sgh/science/2024_minflux/fits/20260106_fitres_NPFit-aGparam_{ct}.h5')

## Profiler
Estimate credible intervals for point estimates from profile likelihood. __Attention: computationally expensive__

This can also move the point estimate, if we find better parameters while exploring

In [9]:
fitres = nl.io.load.hdf5(f'/data/sgh/science/2024_minflux/fits/20260106_fitres_NPFit-aGparam_{ct}.h5')
mci = {}
for treatment in ['ctrl']:
    print()
    print(17*'=')
    print(f'|| {ct:>5s} {treatment:<5s} ||')
    print(17*'=')
    print()
    
    mci[treatment] = {}
    for name in fits[treatment]['joints']:
        print(name)
        print('='*20)
        
        profiler = bayesmsd.Profiler(fits[treatment]['joints'][name], max_restarts=50)
        profiler.point_estimate = fitres[treatment][name]

        with nl.Parallelize():
            mci[treatment][name] = profiler.find_MCI(show_progress=True)

        for key in mci[treatment][name]:
            m, (cil, cih) = mci[treatment][name][key]
            print(f"{key:>25s} = {m:>6.3f} [{cil:>6.3f}, {cih:>6.3f}]")
        print()


||  RH30 ctrl  ||

minflux


profiler iterations: 80it [15:02, 11.28s/it]


  minflux log(σ²) (dim 0) = -8.340 [-8.368, -8.313]
                        α =  0.266 [ 0.254,  0.277]
                  log(αΓ) = -7.055 [-7.117, -6.995]

SPT 100ms


profiler iterations: 83it [11:32,  8.34s/it]


SPT-100ms log(σ²) (dim 0) = -7.034 [-7.053, -7.014]
                        α =  0.356 [ 0.345,  0.367]
                  log(αΓ) = -6.568 [-6.586, -6.550]

SPT 2s


...

   SPT-2s log(σ²) (dim 0) = -6.142 [-6.179, -6.120]
                        α =  0.541 [ 0.523,  0.559]
                  log(αΓ) = -7.027 [-7.050, -6.993]

SPT


profiler iterations: 172it [1:03:40, 22.21s/it]


SPT-100ms log(σ²) (dim 0) = -7.035 [-7.048, -7.024]
   SPT-2s log(σ²) (dim 0) = -7.820 [-7.930, -7.715]
                        α =  0.341 [ 0.336,  0.345]
                  log(αΓ) = -6.614 [-6.621, -6.608]

minflux + SPT


profiler iterations: 226it [2:49:20, 44.96s/it] 

  minflux log(σ²) (dim 0) = -8.219 [-8.231, -8.206]
SPT-100ms log(σ²) (dim 0) = -7.016 [-7.025, -7.006]
   SPT-2s log(σ²) (dim 0) = -7.688 [-7.777, -7.605]
                        α =  0.346 [ 0.343,  0.348]
                  log(αΓ) = -6.624 [-6.629, -6.618]



In [10]:
nl.io.write.hdf5(mci, f'/data/sgh/science/2024_minflux/fits/20260106_fitres_NPFit-aGparam_{ct}.h5')